# Window Detection and Permutation ANOVA

## For Best Frans

In [22]:
from analyses.response_window_analysis import run_permutation_anova_by_window
from analyses.spike_count import extract_spike_counts_from_windows
import pandas as pd
from analyses.response_window_finder.threshold_window_detection import compute_timebinned_spikecount_per_neuron, \
    z_score, threshold_and_fill_gap, extract_consecutive_ranges, remove_consecutive_tuples, \
    find_corresponding_values_for_index_ranges
from tqdm import tqdm
import numpy as np
from analyses.data_readers.recording_metadata_reader import RecordingMetadataReader

In [ ]:
# Detect windows for best frans
prelim = RecordingMetadataReader().get_metadata_for_preliminary_analysis()
results = []
bin_size = 0.05  # in sec
rounded_time = np.round(np.arange(bin_size, 3.50, bin_size), 2)
monkey_group = 'Best Frans'
for _, row in tqdm(prelim.iterrows(), total = len(prelim), desc = "Processing each recording day..."):
    round_no = row['Round No.']
    date = str(row['Date'].strftime('%Y-%m-%d'))
    timebin_spikecount_list = compute_timebinned_spikecount_per_neuron(date, round_no, bin_size, monkey_group)
    for _, r in timebin_spikecount_list.iterrows():
        data = r['TotalSpikeCountList']
        neuron = r['NeuronID']
        normalized_data = z_score(data)
        thresh = 0.5
        change_points = threshold_and_fill_gap(normalized_data, thresh)
        windows = extract_consecutive_ranges(change_points)
        filtered_windows = remove_consecutive_tuples(windows)
        time_windows = find_corresponding_values_for_index_ranges(filtered_windows, rounded_time)
        if len(time_windows) > 0:
            for start_time, end_time in time_windows:
                results.append({
                    'NeuronID': neuron,
                    'WindowStart_ms': int(start_time * 1000),
                    'WindowEnd_ms': int(end_time * 1000)
                })


In [ ]:
# Detected windows
bf_results_df = pd.DataFrame(results)
bf_results_df = bf_results_df.sort_values(by=['NeuronID'])
bf_results_df

In [ ]:
# Extract spike counts in the windows
bf_final_df = extract_spike_counts_from_windows(bf_results_df)

In [ ]:
# Run Perm ANOVA on BestFrans
bestfrans_df = bf_final_df[bf_final_df['MonkeyGroup'] == 'Best Frans']
bf_perm_results, bf_sig_results = run_permutation_anova_by_window(bestfrans_df,
                                category_col='MonkeyName',
                                neuron_col='NeuronID',
                                count_col='SpikeCount',
                                window_start_col='WindowStart_ms',
                                window_end_col='WindowEnd_ms',
                                n_permutations=1000,
                                alpha=0.05,
                                plot=False)

In [ ]:
bf_sig_results

In [ ]:
bf_sig_results[['Date', 'Round No.', 'Cell']] = bf_sig_results['NeuronID'].str.split('_', n=2, expand=True)
bf_sig_results['Time Window'] = list(zip(bf_sig_results['WindowStart_ms'], bf_sig_results['WindowEnd_ms']))
bf_sig_results.head() # --- save this and share with Ed !!

In [ ]:
bf_sig_results.to_excel('bestfrans_panova_passed_cells_after_window_detection_updated.xlsx', index=False)

In [ ]:
# Get spike counts anova passed windows
bf_spike_count_for_sig_windows_anova_passed = extract_spike_counts_from_windows(bf_sig_results)

In [ ]:
bf_spike_count = bf_spike_count_for_sig_windows_anova_passed.copy()
bf_spike_count.head()

In [ ]:
bf_spike_count[['Location','Date', 'Round No.', 'Cell']] = bf_spike_count['NeuronID'].str.split('_', n=3, expand=True)
bf_spike_count['Time Window'] = list(zip(bf_spike_count['WindowStart_ms'], bf_spike_count['WindowEnd_ms']))
bf_spike_count.head()

In [ ]:
group_cols = ['Date', 'Round No.', 'Cell', 'Time Window','MonkeyName']
bf_spike_count['Date'] = bf_spike_count['Date'].astype(str)
bf_grouped_df = bf_spike_count.groupby(group_cols)['SpikeCount'].apply(list).reset_index()
bf_final_spike_count_df = bf_grouped_df.pivot(
    index=['Date', 'Round No.', 'Cell', 'Time Window'],
    columns='MonkeyName',
    values='SpikeCount'
).reset_index()
print(bf_final_spike_count_df) # -- save this and share with Ed!!

In [ ]:
bf_final_spike_count_df.to_excel('bestfrans_spike_counts_for_all_panova_passed_time_windowed_updated.xlsx', index=False)

### For Zombies

In [ ]:
# Detect windows for zombies
prelim = RecordingMetadataReader().get_metadata_for_preliminary_analysis()
results = []
bin_size = 0.05  # in sec
rounded_time = np.round(np.arange(bin_size, 3.50, bin_size), 2)
monkey_group = 'Zombies'
for _, row in tqdm(prelim.iterrows(), total = len(prelim), desc = "Processing each recording day..."):
    round_no = row['Round No.']
    date = str(row['Date'].strftime('%Y-%m-%d'))
    timebin_spikecount_list = compute_timebinned_spikecount_per_neuron(date, round_no, bin_size, monkey_group)
    for _, r in timebin_spikecount_list.iterrows():
        data = r['TotalSpikeCountList']
        neuron = r['NeuronID']
        normalized_data = z_score(data)
        thresh = 0.5
        change_points = threshold_and_fill_gap(normalized_data, thresh)
        windows = extract_consecutive_ranges(change_points)
        filtered_windows = remove_consecutive_tuples(windows)
        time_windows = find_corresponding_values_for_index_ranges(filtered_windows, rounded_time)
        if len(time_windows) > 0:
            for start_time, end_time in time_windows:
                results.append({
                    'NeuronID': neuron,
                    'WindowStart_ms': int(start_time * 1000),
                    'WindowEnd_ms': int(end_time * 1000)
                })


In [ ]:
# Detected windows
zombies_results_df = pd.DataFrame(results)
zombies_results_df = zombies_results_df.sort_values(by=['NeuronID'])
zombies_results_df.head()

In [ ]:
# Extract spike counts in the windows
zombies_extracted_spikes_df = extract_spike_counts_from_windows(zombies_results_df)

In [ ]:
# Run Perm ANOVA on BestFrans
zombies_df = zombies_extracted_spikes_df[zombies_extracted_spikes_df['MonkeyGroup'] == monkey_group]
_, zom_sig_results = run_permutation_anova_by_window(zombies_df,
                                category_col='MonkeyName',
                                neuron_col='NeuronID',
                                count_col='SpikeCount',
                                window_start_col='WindowStart_ms',
                                window_end_col='WindowEnd_ms',
                                n_permutations=1000,
                                alpha=0.05,
                                plot=False)

In [ ]:
zom_sig_results.head()

In [ ]:
zom_sig_copy= zom_sig_results.copy()
zom_sig_copy[['Date', 'Round No.', 'Cell']] = zom_sig_results['NeuronID'].str.split('_', n=2, expand=True)
zom_sig_copy['Time Window'] = list(zip(zom_sig_copy['WindowStart_ms'], zom_sig_copy['WindowEnd_ms']))
zom_sig_copy.head() # --- save this and share with Ed !!

In [ ]:
zom_sig_copy.to_excel('zombies_panova_passed_cells_after_window_detection_updated.xlsx', index=False)
print('Done!')

In [ ]:
# Get spike counts anova passed windows
zom_spike_count_for_sig_windows_anova_passed = extract_spike_counts_from_windows(zom_sig_copy)

In [ ]:
zom_spike_count = zom_spike_count_for_sig_windows_anova_passed.copy()
zom_spike_count.head()

In [ ]:
zom_spike_count[['Location','Date', 'Round No.', 'Cell']] = zom_spike_count['NeuronID'].str.split('_', n=3, expand=True)
zom_spike_count['Time Window'] = list(zip(zom_spike_count['WindowStart_ms'], zom_spike_count['WindowEnd_ms']))
zom_spike_count.head()

In [ ]:
zom_spike_count.head()
group_cols = ['Date', 'Round No.', 'Cell', 'Time Window', 'MonkeyName']
zom_spike_count['Date'] = zom_spike_count['Date'].astype(str)
zom_grouped_df = zom_spike_count.groupby(group_cols)['SpikeCount'].apply(list).reset_index()
zom_final_spike_count_df = zom_grouped_df.pivot(
    index=['Date', 'Round No.', 'Cell', 'Time Window'],
    columns='MonkeyName',
    values='SpikeCount'
).reset_index()
print(zom_final_spike_count_df.head())  # -- save this and share with Ed!!

In [ ]:
zom_final_spike_count_df.to_excel('zombies_spike_counts_for_all_panova_passed_time_windowed_updated.xlsx', index=False)